# Ensemble Recommendation Model

Blend multiple recommendation models for improved performance.

**Strategy:**
- Load predictions from trained base models
- Blend using weighted ranking score fusion
- Register ensemble model to Unity Catalog

**Base Models:**
1. Popularity Model - Time-based trending items
2. Age Rules Model - Age segment preferences

**Expected Performance:** MAP@12 > individual models

## Setup

In [0]:
import sys

# Add project root to path (go up 2 levels from notebooks/)
sys.path.append("../../")

from pyspark.sql.functions import *
from pyspark.sql.types import *
import pandas as pd
import mlflow
import mlflow.pyfunc

from config.catalog_config import *
from config.model_config import EVAL_CONFIG, ENSEMBLE_CONFIG
from config.widget_utils import get_bundle_parameters
from utils.data_utils import load_delta_table
from utils.evaluation_utils import calculate_map_at_k, log_evaluation_metrics

In [0]:
# Get parameters from bundle (passed as notebook parameters)
# Falls back to defaults when running interactively
params = get_bundle_parameters(model_default="ensemble_model")
catalog_name = params["catalog_name"]
schema_name = params["schema_name"]
experiment_name = params["experiment_name"]
model_name = params["model_name"]

print(f"Parameters:")
print(f"  Catalog: {catalog_name}")
print(f"  Schema: {schema_name}")
print(f"  Experiment: {experiment_name}")
print(f"  Model: {model_name}")

In [0]:
# Initialize MLflow
mlflow.set_experiment(experiment_name)

## Configuration

In [0]:
# Ensemble configuration - 3 models with weighted blending
weights = ENSEMBLE_CONFIG["weights"]
popularity_weight = weights["popularity"]
age_rules_weight = weights["age_rules"]
lstm_weight = weights["lstm"]

# Model name mapping
MODEL_NAMES = {
    "popularity": f"{catalog_name}.{schema_name}.popularity_model",
    "age_rules": f"{catalog_name}.{schema_name}.age_rules_model",
    "lstm": f"{catalog_name}.{schema_name}.lstm_model"
}

print("Ensemble Configuration:")
print(f"  Base Models: {list(weights.keys())}")
print(f"  Weights: popularity={popularity_weight}, age_rules={age_rules_weight}, lstm={lstm_weight}")
print(f"\nModel Names:")
for model_type, model_name in MODEL_NAMES.items():
    print(f"  {model_type}: {model_name}")

## Load Data

In [0]:
print("Loading data...")

# Load validation ground truth for evaluation
val_ground_truth = load_delta_table(f"{catalog_name}.{schema_name}.val_ground_truth_silver")
customers_df = load_delta_table(f"{catalog_name}.{schema_name}.customers_bronze")

print(f"Validation customers: {val_ground_truth.count():,}")
print(f"Total customers: {customers_df.count():,}")

## Load Base Model Predictions

In [0]:
print("Loading base model predictions...")

# Load predictions from each model (saved during training)
popularity_preds = load_delta_table(f"{catalog_name}.{schema_name}.popularity_predictions_gold")
age_rules_preds = load_delta_table(f"{catalog_name}.{schema_name}.age_rules_predictions_gold")
lstm_preds = load_delta_table(f"{catalog_name}.{schema_name}.lstm_predictions_gold")

print(f"Popularity predictions: {popularity_preds.count():,}")
print(f"Age rules predictions: {age_rules_preds.count():,}")
print(f"LSTM predictions: {lstm_preds.count():,}")

## Blend Predictions

In [0]:
print("Blending predictions using weighted ranking strategy...")

# Create blending UDF for 3 models
@udf(ArrayType(IntegerType()))
def blend_recommendations_udf(pop_articles, age_articles, lstm_articles):
    """
    Blend recommendations from 3 models using weighted ranking

    Args:
        pop_articles: Popularity model recommendations
        age_articles: Age rules model recommendations
        lstm_articles: LSTM model recommendations

    Returns:
        List of blended article IDs
    """
    if not pop_articles:
        pop_articles = []
    if not age_articles:
        age_articles = []
    if not lstm_articles:
        lstm_articles = []

    # Score each article based on its rank in each model's predictions
    article_scores = {}

    # Popularity model (reciprocal rank scoring)
    for rank, article in enumerate(pop_articles[:12]):
        score = popularity_weight * (1.0 / (rank + 1))
        article_scores[article] = article_scores.get(article, 0) + score

    # Age rules model
    for rank, article in enumerate(age_articles[:12]):
        score = age_rules_weight * (1.0 / (rank + 1))
        article_scores[article] = article_scores.get(article, 0) + score

    # LSTM model
    for rank, article in enumerate(lstm_articles[:12]):
        score = lstm_weight * (1.0 / (rank + 1))
        article_scores[article] = article_scores.get(article, 0) + score

    # Sort by score and return top 12
    sorted_articles = sorted(article_scores.items(), key=lambda x: x[1], reverse=True)
    return [int(article) for article, score in sorted_articles[:12]]

In [0]:
# Join predictions from all 3 models
print("Joining predictions from all 3 models...")

ensemble_df = (
    popularity_preds.select(
        col("customer_id"),
        col("predicted_articles").alias("pop_articles")
    )
    .join(
        age_rules_preds.select(
            col("customer_id"),
            col("predicted_articles").alias("age_articles")
        ),
        "customer_id",
        "inner"
    )
    .join(
        lstm_preds.select(
            col("customer_id"),
            col("predicted_articles").alias("lstm_articles")
        ),
        "customer_id",
        "inner"
    )
)

print(f"Joined predictions: {ensemble_df.count():,}")

In [0]:
# Apply blending
ensemble_predictions = ensemble_df.withColumn(
    "predicted_articles",
    blend_recommendations_udf(col("pop_articles"), col("age_articles"), col("lstm_articles"))
).select("customer_id", "predicted_articles")

print(f"Ensemble predictions: {ensemble_predictions.count():,}")

# Show samples
print("\nSample ensemble predictions:")
display(ensemble_predictions.limit(5))

## Evaluate Ensemble Model

In [0]:
# ========== PREPARE MLFLOW DATASETS (MLflow 3.0+) ==========
print("\n" + "="*60)
print("PREPARING MLFLOW DATASETS")
print("="*60)

# Create MLflow datasets for tracking
ensemble_predictions_pd = ensemble_predictions.limit(10000).toPandas()
val_ground_truth_pd = val_ground_truth.limit(10000).toPandas()

ensemble_dataset = mlflow.data.from_pandas(ensemble_predictions_pd, name="ensemble_predictions")
val_dataset = mlflow.data.from_pandas(val_ground_truth_pd, name="validation")

print("✓ Datasets created for MLflow tracking")
print(f"  Ensemble dataset: {len(ensemble_predictions_pd):,} samples")
print(f"  Val dataset: {len(val_ground_truth_pd):,} samples")

# ========== START MLFLOW RUN WITH SYSTEM METRICS (MLflow 3.0+) ==========
print("\n" + "="*60)
print("STARTING MLFLOW 3.0+ RUN WITH SYSTEM METRICS")
print("="*60)

print("Evaluating ensemble model...")

# Start MLflow run with system metrics logging enabled
with mlflow.start_run(run_name="ensemble_blend", log_system_metrics=True) as run:
    
    run_id = run.info.run_id
    print("✓ System metrics logging enabled (CPU, GPU, memory, network, disk)")

    # Log parameters
    mlflow.log_param("model_type", "ensemble")
    mlflow.log_param("blend_strategy", "weighted_ranking")
    mlflow.log_param("weight_popularity", popularity_weight)
    mlflow.log_param("weight_age_rules", age_rules_weight)
    mlflow.log_param("weight_lstm", lstm_weight)
    mlflow.log_param("num_base_models", 3)
    mlflow.log_param("base_models", "popularity,age_rules,lstm")

    mlflow.set_tag("stage", "training")
    mlflow.set_tag("model_type", "ensemble")
    mlflow.set_tag("mlflow_version", "3.0+")
    mlflow.set_tag("features", "system_metrics,dataset_linking")

    # Evaluate on validation set
    print("\n" + "="*60)
    print("EVALUATING ENSEMBLE MODEL")
    print("="*60)

    metrics = log_evaluation_metrics(
        ensemble_predictions,
        val_ground_truth,
        "Ensemble Model",
        k=12
    )

    # ========== LOG METRICS LINKED TO DATASET (MLflow 3.0+) ==========
    # Log metrics with dataset linking
    mlflow.log_metric(
        key="map@12",
        value=metrics['map@12'],
        dataset=val_dataset,
    )
    mlflow.log_metric(
        key="num_customers",
        value=metrics['num_customers'],
        dataset=val_dataset,
    )
    if "catalog_coverage" in metrics:
        mlflow.log_metric(
            key="catalog_coverage",
            value=metrics['catalog_coverage'],
            dataset=val_dataset,
        )

    # Save run ID
    print(f"\nMLflow run ID: {run_id}")

## Compare with Base Models

In [0]:
print("Comparing ensemble with base models...")

# Evaluate each base model
print("\n" + "="*60)
print("MODEL COMPARISON")
print("="*60)

comparison_results = []

# Popularity
pop_map12 = calculate_map_at_k(
    popularity_preds.select("customer_id", "predicted_articles"),
    val_ground_truth,
    k=12
)
comparison_results.append(("Popularity", pop_map12))
print(f"Popularity Model: MAP@12 = {pop_map12:.6f}")

# Age Rules
age_map12 = calculate_map_at_k(
    age_rules_preds.select("customer_id", "predicted_articles"),
    val_ground_truth,
    k=12
)
comparison_results.append(("Age Rules", age_map12))
print(f"Age Rules Model: MAP@12 = {age_map12:.6f}")

# LSTM
lstm_map12 = calculate_map_at_k(
    lstm_preds.select("customer_id", "predicted_articles"),
    val_ground_truth,
    k=12
)
comparison_results.append(("LSTM", lstm_map12))
print(f"LSTM Model: MAP@12 = {lstm_map12:.6f}")

# Ensemble
comparison_results.append(("Ensemble", metrics["map@12"]))
print(f"Ensemble Model: MAP@12 = {metrics['map@12']:.6f}")

In [0]:
# Visualize comparison
import matplotlib.pyplot as plt

comparison_df = pd.DataFrame(comparison_results, columns=["Model", "MAP@12"])
comparison_df = comparison_df.sort_values("MAP@12", ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(comparison_df["Model"], comparison_df["MAP@12"])

# Color the best model
best_idx = comparison_df["MAP@12"].idxmax()
bars[best_idx].set_color("green")

ax.set_xlabel("MAP@12 Score")
ax.set_title("Model Performance Comparison")
ax.grid(axis="x", alpha=0.3)

# Add value labels
for i, v in enumerate(comparison_df["MAP@12"]):
    ax.text(v + 0.0005, i, f"{v:.6f}", va="center")

plt.tight_layout()
display(fig)

# Log to MLflow
with mlflow.start_run(run_name="ensemble_comparison"):
    mlflow.log_figure(fig, "model_comparison.png")
    mlflow.log_dict(comparison_df.to_dict(orient="records"), "comparison_metrics.json")
    mlflow.set_tag("analysis_type", "model_comparison")

In [0]:
# Print comparison table
print("\n" + "="*60)
print("FINAL MODEL COMPARISON")
print("="*60)
for model, score in comparison_results:
    is_best = " ⭐ BEST" if score == max([s for _, s in comparison_results]) else ""
    print(f"{model:20s}: MAP@12 = {score:.6f}{is_best}")
print("="*60)

## Create Ensemble Model Wrapper for Registry

### Helper Functions

In [ ]:
def blend_recommendations(
    customer_id,
    pop_articles,
    age_articles,
    lstm_articles,
    weights_dict,
    top_k=12
):
    """
    Blend recommendations from multiple models using weighted ranking
    
    Args:
        customer_id: Customer identifier
        pop_articles: Popularity model recommendations (list of article IDs)
        age_articles: Age rules model recommendations (list of article IDs)
        lstm_articles: LSTM model recommendations (list of article IDs)
        weights_dict: Dictionary with weights for each model
        top_k: Number of recommendations to return
        
    Returns:
        List of top_k blended article IDs
    """
    if not pop_articles:
        pop_articles = []
    if not age_articles:
        age_articles = []
    if not lstm_articles:
        lstm_articles = []
    
    # Score each article based on its rank in each model's predictions
    article_scores = {}
    
    # Popularity model (reciprocal rank scoring)
    for rank, article in enumerate(pop_articles[:top_k]):
        score = weights_dict.get("popularity", 0) * (1.0 / (rank + 1))
        article_scores[article] = article_scores.get(article, 0) + score
    
    # Age rules model
    for rank, article in enumerate(age_articles[:top_k]):
        score = weights_dict.get("age_rules", 0) * (1.0 / (rank + 1))
        article_scores[article] = article_scores.get(article, 0) + score
    
    # LSTM model
    for rank, article in enumerate(lstm_articles[:top_k]):
        score = weights_dict.get("lstm", 0) * (1.0 / (rank + 1))
        article_scores[article] = article_scores.get(article, 0) + score
    
    # Sort by score and return top K
    sorted_articles = sorted(article_scores.items(), key=lambda x: x[1], reverse=True)
    return [article for article, score in sorted_articles[:top_k]]

print("✓ Helper function defined: blend_recommendations")


In [0]:
# Create PyFunc wrapper for ensemble model
class EnsembleModelWrapper(mlflow.pyfunc.PythonModel):
    """
    MLflow PyFunc wrapper for ensemble recommendation model
    Loads and blends predictions from multiple registered models
    """

    def __init__(self, model_versions, weights):
        """
        Args:
            model_versions: Dict mapping model_name -> version
            weights: Dict mapping model_name -> weight
        """
        self.model_versions = model_versions
        self.weights = weights
        self.models = {}

    def load_context(self, context):
        """Load all base models from MLflow Model Registry"""
        print("Loading base models...")

        for model_type, model_name in MODEL_NAMES.items():
            if model_type in self.model_versions:
                version = self.model_versions[model_type]
                model_uri = f"models:/{model_name}/{version}"
                print(f"  Loading {model_name} v{version}...")
                try:
                    self.models[model_type] = mlflow.pyfunc.load_model(model_uri)
                except Exception as e:
                    print(f"  Warning: Could not load {model_name}: {e}")

        print(f"Loaded {len(self.models)} base models")

    def predict(self, context, model_input):
        """
        Generate ensemble predictions

        Args:
            model_input: DataFrame with customer_id (and age_group if using age_rules)

        Returns:
            DataFrame with customer_id and predicted_articles
        """
        if not isinstance(model_input, pd.DataFrame):
            raise ValueError("Input must be a pandas DataFrame")

        if "customer_id" not in model_input.columns:
            raise ValueError("Input must contain 'customer_id' column")

        # Get predictions from each model
        all_predictions = {}

        # Popularity model
        if "popularity" in self.models:
            try:
                pop_preds = self.models["popularity"].predict(model_input[["customer_id"]])
                all_predictions["popularity"] = dict(
                    zip(pop_preds["customer_id"], pop_preds["predicted_articles"])
                )
            except Exception as e:
                print(f"Popularity model prediction failed: {e}")

        # Age rules model (needs age_group)
        if "age_rules" in self.models and "age_group" in model_input.columns:
            try:
                age_preds = self.models["age_rules"].predict(
                    model_input[["customer_id", "age_group"]]
                )
                all_predictions["age_rules"] = dict(
                    zip(age_preds["customer_id"], age_preds["predicted_articles"])
                )
            except Exception as e:
                print(f"Age rules model prediction failed: {e}")

        # LSTM model (needs purchase sequence - would need to load from database)
        # For simplicity, we skip LSTM in real-time inference
        # In production, you'd fetch customer sequences and run LSTM inference

        # Blend predictions
        results = []
        for _, row in model_input.iterrows():
            customer_id = row["customer_id"]

            pop_articles = all_predictions.get("popularity", {}).get(customer_id, [])
            age_articles = all_predictions.get("age_rules", {}).get(customer_id, [])
            lstm_articles = []  # TODO: Add LSTM inference if sequences available

            blended = blend_recommendations(
                customer_id,
                pop_articles,
                age_articles,
                lstm_articles,
                self.weights,
                top_k=12
            )

            results.append({"customer_id": customer_id, "predicted_articles": blended})

        return pd.DataFrame(results)


In [0]:
# Get latest versions of each model
from mlflow.tracking import MlflowClient

client = MlflowClient()

model_versions = {}
for model_type, model_name in MODEL_NAMES.items():
    try:
        versions = client.search_model_versions(f"name='{model_name}'")
        if versions:
            latest_version = max([int(v.version) for v in versions])
            model_versions[model_type] = latest_version
            print(f"{model_name}: v{latest_version}")
    except Exception as e:
        print(f"Could not find {model_name}: {e}")

print(f"\nUsing model versions: {model_versions}")

## Register Ensemble Model

In [0]:
print("Registering ensemble model to MLflow Model Registry...")

with mlflow.start_run(run_id=run_id):
    # Create ensemble model instance
    ensemble_model = EnsembleModelWrapper(
        model_versions=model_versions,
        weights=weights
    )

    # Define signature (for inference without LSTM)
    from mlflow.models.signature import infer_signature

    sample_input = pd.DataFrame({
        "customer_id": ["sample_1", "sample_2"],
        "age_group": ["25-34", "35-44"]
    })

    # For signature, we'll use a simplified version that doesn't actually load models
    sample_output = pd.DataFrame({
        "customer_id": ["sample_1", "sample_2"],
        "predicted_articles": [[], []]
    })
    signature = infer_signature(sample_input, sample_output)

    # ========== LOG MODEL (MLflow 3.0+) ==========
    # Log model with name, params, and input_example
    model_info = mlflow.pyfunc.log_model(
        artifact_path="ensemble_model",
        name="ensemble-model",
        python_model=ensemble_model,
        params={
            "model_type": "ensemble",
            "blend_strategy": "weighted_ranking",
            "weight_popularity": popularity_weight,
            "weight_age_rules": age_rules_weight,
            "weight_lstm": lstm_weight,
            "num_base_models": 3,
        },
        signature=signature,
        input_example=sample_input,
    )
    
    print(f"✓ Model logged with model_id: {model_info.model_id}")

# Register to Unity Catalog Model Registry
uc_model_name = f"{catalog_name}.{schema_name}.{model_name}"
model_uri = f"runs:/{run_id}/ensemble_model"

print(f"\nRegistering ensemble model to Unity Catalog: {uc_model_name}")

registered_model = mlflow.register_model(model_uri, uc_model_name)

print(f"✓ Model registered: {uc_model_name}")
print(f"✓ Version: {registered_model.version}")
print(f"✓ Run ID: {run_id}")

In [0]:
# Add model description
from mlflow.tracking import MlflowClient

client = MlflowClient()
client.update_registered_model(
    name=uc_model_name,
    description=f"Ensemble recommendation model blending 3 base models. "
    f"MAP@12: {metrics['map@12']:.6f}. "
    f"Weighted ranking strategy with popularity={popularity_weight}, age_rules={age_rules_weight}, lstm={lstm_weight}.",
)

client.update_model_version(
    name=uc_model_name,
    version=registered_model.version,
    description=f"Blend of popularity, age_rules, and LSTM models. "
    f"MAP@12: {metrics['map@12']:.6f}. Weighted ranking with reciprocal rank scoring.",
)

print(f"\n✓ Ensemble model {uc_model_name} v{registered_model.version} registered successfully!")

## Save Predictions

In [0]:
# Save ensemble predictions
output_table = f"{catalog_name}.{schema_name}.ensemble_predictions_gold"
print(f"Saving predictions to {output_table}...")

ensemble_predictions \
    .withColumn("model_type", lit("ensemble")) \
    .withColumn("run_id", lit(run_id)) \
    .write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(output_table)

print(f"✓ Predictions saved to {output_table}")

## Summary

In [0]:
print("\n" + "="*60)
print("ENSEMBLE MODEL TRAINING COMPLETE")
print("="*60)
print(f"Strategy: Weighted ranking blend")
print(f"Base Models: {len(model_versions)}")
for model_type, version in model_versions.items():
    weight = weights[model_type]
    print(f"  - {model_type}: v{version} (weight={weight})")
print(f"")
print(f"Performance:")
print(f"  Ensemble MAP@12: {metrics['map@12']:.6f}")
print(f"  Improvement over best base model: "
      f"{(metrics['map@12'] - max([s for _, s in comparison_results[:-1]])):.6f}")
print(f"")
print(f"MLflow:")
print(f"  Model: {model_name} v{registered_model.version}")
print(f"  Run ID: {run_id}")
print(f"  Predictions: {output_table}")
print("="*60)
print("\n✓ Sprint 4 Complete! All models trained and ensemble deployed.")
print("="*60)